In [1]:
import torch, torchio as tio
import torch.nn.functional as F

from torch.utils.data import DataLoader
from datasets import *
from models import *
from pathlib import Path
from torch.amp import autocast
from sklearn.model_selection import train_test_split

/nfs/ForHenry/tensor_metric_infonce_training/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
@torch.inference_mode()
def knn_self_accuracy(
    encoder, loader: DataLoader, k=5, device="cuda", max_batches=None
):
    encoder.eval()
    z1_list, z2_list = [], []

    for b, (x1, x2) in enumerate(loader, 1):
        if max_batches and b > max_batches:
            break
        x1 = x1.to(device, non_blocking=True)
        x2 = x2.to(device, non_blocking=True)

        with autocast(device.type):
            z1 = F.normalize(encoder(x1), dim=1)
            z2 = F.normalize(encoder(x2), dim=1)
        z1_list.append(z1)
        z2_list.append(z2)

    z1_all = torch.cat(z1_list, dim=0)
    z2_all = torch.cat(z2_list, dim=0)
    z = torch.cat([z1_all, z2_all], dim=0)

    sim = torch.mm(z, z.t())
    sim.fill_diagonal_(-1e9)

    _, idx = sim.topk(k, dim=1)  # (2N, k)

    N = z1_all.size(0)
    ar = torch.arange(2 * N, device=z.device)
    pos_idx = torch.where(ar < N, ar + N, ar - N)  # (2N,)

    hits = (idx == pos_idx.view(-1, 1)).any(dim=1)
    topk_acc = hits.float().mean().item()
    return topk_acc

In [4]:
home_dir = "/nfs/ForHenry/tensor_metric_infonce_training"

input_file = f"{home_dir}/data/tensor_paths.txt"
with open(input_file, "r") as f:
    final_paths = [Path(line.strip()) for line in f if line.strip()]

print(f"Loaded {len(final_paths)} paths from {input_file}")

_, val_paths = train_test_split(final_paths, test_size=0.2, random_state=42)

Loaded 23102 paths from /nfs/ForHenry/tensor_metric_infonce_training/data/tensor_paths.txt


In [5]:
val_dataset = TensorDataset(path_list=val_paths)
val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=8,
    pin_memory=True,
)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# encoder = ResidualSEEncoder(layers=(3, 4, 6, 3), num_channels=6)
encoder = ResidualSEEncoder()
encoder.to(device)
with torch.no_grad():
    dummy = torch.randn(2, 6, 64, 64, 64, device=device)
    _ = encoder(dummy)
ckpt = torch.load(
    "outputs/checkpoints/epoch7.pth", map_location=device, weights_only=True
)
encoder.load_state_dict(ckpt["model"] if "model" in ckpt else ckpt)

IndexError: tuple index out of range

In [10]:
top5 = knn_self_accuracy(encoder, val_loader, k=1, device=device, max_batches=64)
print(f"Self k-NN top-1 accuracy: {top5:.3f}")

[Warning] File not found: /nfs2/harmonization/BIDS/EBDT/derivatives/sub-NDARNP627KZ0/ses-6yr/PreQual/TENSOR/dwmri_tensor.nii.gz, trying next index...
[Warning] File not found: /nfs2/harmonization/BIDS/EBDT/derivatives/sub-NDARGG949BMG/ses-2yr/PreQual/TENSOR/dwmri_tensor.nii.gz, trying next index...
[Warning] File not found: /nfs2/harmonization/BIDS/EBDT/derivatives/sub-NDARGE926WV7/ses-4yr/PreQual/TENSOR/dwmri_tensor.nii.gz, trying next index...
[Warning] File not found: /nfs2/harmonization/BIDS/EBDT/derivatives/sub-NDARHW893NAW/ses-6yr/PreQual/TENSOR/dwmri_tensor.nii.gz, trying next index...
[Warning] File not found: /nfs2/harmonization/BIDS/EBDT/derivatives/sub-NDARTR592ZPA/ses-2yr/PreQual/TENSOR/dwmri_tensor.nii.gz, trying next index...
[Warning] File not found: /nfs2/harmonization/BIDS/EBDT/derivatives/sub-NDARCR399LVB/ses-2yr/PreQual/TENSOR/dwmri_tensor.nii.gz, trying next index...
[Warning] File not found: /nfs2/harmonization/BIDS/EBDT/derivatives/sub-NDARNR910MJ8/ses-6yr/PreQual